# ⚡ Análisis de Generación Eléctrica en México (SEN) - Año 2025
**Metodología Refactorizada y Visualización Institucional (Ref: CONAHCYT PLANEAS)**

Este cuaderno implementa una ruta de extracción dinámica (sin rutas absolutas fijas) que localiza automáticamente las liquidaciones del Sistema Eléctrico Nacional (SEN). Selecciona siempre la liquidación de máxima prioridad disponible (`L3 > L2 > L1 > L0`), aplica la corrección matemática de la 'Hora 24' y genera visualizaciones con paletas armonizadas inspiradas en la Plataforma Nacional de Energía, Ambiente y Sociedad (PLANEAS).

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================================================================
# 1. CONFIGURACIÓN DINÁMICA DE RUTAS (REFACTORIZADO)
# ==============================================================================
# Búsqueda automática en el directorio de trabajo actual y sus subcarpetas
BASE_DIR = os.path.abspath(".")
RUTA_SALIDA_CSV = os.path.join(BASE_DIR, "Consolidado_Energias_Generadas.csv")

print("📁 Buscando archivos de generación eléctrica de forma dinámica...")
archivos_csv = glob.glob(os.path.join(BASE_DIR, "**", "Generacion Liquidada*.csv"), recursive=True)

archivos_por_mes = {}
for ruta in archivos_csv:
    nombre = os.path.basename(ruta)
    match = re.search(r"SEN\s+([a-z]+)\s+\d{4}", nombre, re.IGNORECASE)
    if match:
        mes = match.group(1).lower()
        archivos_por_mes.setdefault(mes, []).append(ruta)

print(f"📊 {len(archivos_por_mes)} meses detectados: {list(archivos_por_mes.keys())}")

In [ ]:
# ==============================================================================
# 2. PROCESAMIENTO Y CORRECCIÓN DE HORA 24 (SIMPLIFICADO)
# ==============================================================================
def obtener_prioridad(ruta):
    match = re.search(r"_L(\d+)", os.path.basename(ruta))
    return int(match.group(1)) if match else -1

lista_dataframes = []
tecnologias = [
    "Eolica", "Fotovoltaica", "Biomasa", "Carboelectrica", "Ciclo Combinado",
    "Combustion Interna", "Geotermoelectrica", "Hidroelectrica", "Nucleoelectrica",
    "Termica Convencional", "Turbo Gas"
]

print("⚙️ Procesando liquidaciones con máxima prioridad (L3 > L2 > L1 > L0)...")
for mes, rutas in archivos_por_mes.items():
    ruta_elegida = max(rutas, key=obtener_prioridad)
    
    try:
        df_sen = pd.read_csv(ruta_elegida, skiprows=7, sep=";")
        if len(df_sen.columns) <= 1:
            df_sen = pd.read_csv(ruta_elegida, skiprows=7, sep=",")
            
        df_sen.columns = df_sen.columns.str.strip()
        cols = df_sen.columns
        
        buscar = lambda texto: next((c for c in cols if texto.lower() in c.lower()), None)
        col_dia = buscar("dia")
        col_hora = buscar("hora")
        
        # Corrección matemática de la hora 24
        fecha_base = pd.to_datetime(df_sen[col_dia].astype(str).str.strip(), dayfirst=True, errors='coerce')
        horas_num = pd.to_numeric(df_sen[col_hora], errors='coerce').fillna(1).astype(int)
        
        df_procesado = pd.DataFrame()
        df_procesado["fecha_hora"] = (fecha_base + pd.to_timedelta(horas_num, unit='h')).dt.floor("h")
        
        for tec in tecnologias:
            col_real = buscar(tec)
            df_procesado[tec] = pd.to_numeric(df_sen[col_real], errors='coerce').fillna(0.0) if col_real else 0.0
                
        lista_dataframes.append(df_procesado)
    except Exception as e:
        print(f"   ❌ Error en {os.path.basename(ruta_elegida)}: {e}")

todos_los_datos = pd.concat(lista_dataframes, ignore_index=True)
todos_los_datos = todos_los_datos.sort_values(by="fecha_hora").reset_index(drop=True)
df_graficos = todos_los_datos.copy()

# Guardar CSV consolidado
todos_los_datos["fecha_hora_str"] = todos_los_datos["fecha_hora"].dt.strftime("%d/%m/%Y %H:%M")
todos_los_datos_export = todos_los_datos.rename(columns={"fecha_hora_str": "Fecha Hora"}).drop(columns=["fecha_hora"])
todos_los_datos_export.to_csv(RUTA_SALIDA_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Consolidado guardado con éxito: {RUTA_SALIDA_CSV}")

# 3. Visualizaciones Institucionales (Estilo CONAHCYT PLANEAS)
Se aplica una paleta de colores distintiva que separa fuentes térmicas fósiles (tonos profundos de azul, petróleo y teal) de fuentes renovables y limpias (ámbar, esmeralda, naranja quemado, terra y carmín).

In [ ]:
# Mapeo institucional de colores por tecnología
COLOR_MAP = {
    "Ciclo Combinado": "#003049",       # Azul Oscuro Profundo
    "Termica Convencional": "#005f73",  # Azul Petróleo
    "Carboelectrica": "#0A9396",        # Teal
    "Combustion Interna": "#48cae4",    # Azul Claro
    "Turbo Gas": "#90e0ef",             # Celeste
    "Hidroelectrica": "#2a9d8f",        # Esmeralda
    "Fotovoltaica": "#ee9b00",          # Ámbar
    "Eolica": "#ca6702",                # Naranja Quemado
    "Geotermoelectrica": "#bb3e03",     # Terra
    "Nucleoelectrica": "#ae2012",       # Carmín
    "Biomasa": "#9b5de5"                # Púrpura
}

fosiles = ["Ciclo Combinado", "Combustion Interna", "Carboelectrica", "Termica Convencional", "Turbo Gas"]
renovables = ["Eolica", "Fotovoltaica", "Biomasa", "Geotermoelectrica", "Hidroelectrica", "Nucleoelectrica"]

sns.set_theme(style="white")
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16,        
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'font.family': 'sans-serif',
    'figure.autolayout': True
})

df_graficos = df_graficos[df_graficos['fecha_hora'].dt.year == 2025].copy()
columnas_energia = [c for c in tecnologias if c in df_graficos.columns]

In [ ]:
# --- GRÁFICA 1: PARTICIPACIÓN ANUAL POR TECNOLOGÍA ---
totales_anuales = df_graficos[columnas_energia].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 8))
colores_pie = [COLOR_MAP.get(tec, "#333333") for tec in totales_anuales.index]
wedges, texts, autotexts = ax.pie(
    totales_anuales, autopct=lambda pct: f'{pct:.1f}%' if pct > 1.5 else '',
    startangle=140, colors=colores_pie,
    explode=[0.03 if i < 3 else 0.01 for i in range(len(totales_anuales))],
    pctdistance=0.75, wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2)
)
for idx, auttext in enumerate(autotexts):
    auttext.set_fontsize(11)
    auttext.set_weight('bold')
    auttext.set_color('white' if colores_pie[idx] in ["#003049", "#005f73", "#ae2012", "#ca6702", "#bb3e03", "#6A040F", "#2a9d8f"] else '#003049')

ax.legend(wedges, [f"{t} ({(v/totales_anuales.sum())*100:.1f}%)" for t, v in zip(totales_anuales.index, totales_anuales.values)],
          title="Tecnologías", title_fontproperties={'weight':'bold', 'size': 13},
          loc="center left", bbox_to_anchor=(1, 0, 0.5, 1), fontsize=11, frameon=True)
ax.set_title("Matriz de Generación Eléctrica Anual 2025 (SEN)", weight='bold', pad=15)
plt.show()

In [ ]:
# --- GRÁFICA 1B: FÓSILES VS RENOVABLES Y LIMPIAS ---
gen_fosil = totales_anuales[totales_anuales.index.isin(fosiles)].sum()
gen_renovable = totales_anuales[totales_anuales.index.isin(renovables)].sum()

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    [gen_fosil, gen_renovable], labels=['Fósiles', 'Renovables y Limpias'],
    autopct='%1.1f%%', startangle=90, colors=['#003049', '#2a9d8f'],
    explode=[0.03, 0.03], pctdistance=0.65, wedgeprops=dict(width=0.5, edgecolor='white', linewidth=3)
)
for auttext in autotexts:
    auttext.set_fontsize(14)
    auttext.set_weight('bold')
    auttext.set_color('white')
ax.set_title("Transición Energética: Fósiles vs Limpias (SEN 2025)", weight='bold', pad=15)
plt.show()

In [ ]:
# --- GRÁFICAS 2 Y 3: EVOLUCIÓN MENSUAL Y ÁREA APILADA ---
df_graficos['Mes_Nombre'] = df_graficos['fecha_hora'].dt.strftime('%Y-%m')
df_mensual = df_graficos.groupby('Mes_Nombre')[columnas_energia].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 6.5))
marcadores = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'X']
for idx, col in enumerate(columnas_energia):
    ax.plot(df_mensual['Mes_Nombre'], df_mensual[col] / 1e3, marker=marcadores[idx % len(marcadores)],
            markersize=7, linewidth=2.5, label=col, color=COLOR_MAP.get(col, "#333333"))
ax.set_title("Evolución Mensual de Producción Energética (GWh) - 2025", weight='bold', pad=15)
ax.set_xlabel("Mes del Año", weight='bold', labelpad=10)
ax.set_ylabel("Generación Mensual (GWh)", weight='bold', labelpad=10)
ax.grid(axis='both', linestyle='--', alpha=0.4)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Tecnologías", frameon=True, fontsize=10)
sns.despine()
plt.show()

# Área Apilada
fig, ax = plt.subplots(figsize=(14, 6.5))
ax.stackplot(df_mensual['Mes_Nombre'], [df_mensual[c] / 1e3 for c in columnas_energia],
             labels=columnas_energia, colors=[COLOR_MAP.get(c, "#333333") for c in columnas_energia], alpha=0.9)
ax.set_title("Volumen Acumulado Mensual de Energía Inyectada al SEN (GWh) - 2025", weight='bold', pad=15)
ax.set_xlabel("Mes del Año", weight='bold', labelpad=10)
ax.set_ylabel("Generación Acumulada (GWh)", weight='bold', labelpad=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Tecnologías", fontsize=10)
sns.despine()
plt.show()

In [ ]:
# --- GRÁFICA 4: PERFIL HORARIO PROMEDIO ---
df_graficos['Hora_Dia'] = df_graficos['fecha_hora'].dt.hour
df_horario = df_graficos.groupby('Hora_Dia')[columnas_energia].mean()

fig, ax = plt.subplots(figsize=(14, 6.5))
ax.stackplot(df_horario.index, [df_horario[c] for c in columnas_energia],
             labels=columnas_energia, colors=[COLOR_MAP.get(c, "#333333") for c in columnas_energia], alpha=0.9)
ax.set_title("Curva de Despacho Promedio Diaria por Tecnología (SEN 2025)", weight='bold', pad=15)
ax.set_xlabel("Hora del Día (00:00 - 23:00)", weight='bold', labelpad=10)
ax.set_ylabel("Demanda Cubierta Promedio (MWh)", weight='bold', labelpad=10)
ax.set_xlim(0, 23)
ax.set_xticks(range(0, 24))
ax.grid(axis='both', linestyle='--', alpha=0.3)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Tecnologías", fontsize=10)
sns.despine()
plt.show()